# I. Import Library


In [1]:
!pip install py7zr
import pandas as pd
import numpy as np
import os
import shutil
import py7zr
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 10.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 6.7 MB/s eta 0:00:00


# II. Load Dataset
## 1. Convert to CSV


In [2]:

input_dir = "/kaggle/input/competitions/favorita-grocery-sales-forecasting/"
output_dir = "/kaggle/working/extractedCSV"

# 1. Clean previous extraction (prevents duplication issues)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)

# 2. Extract all .7z files
for file in os.listdir(input_dir):
    if file.endswith(".7z"):
        file_path = os.path.join(input_dir, file)

        with py7zr.SevenZipFile(file_path, 'r') as archive:
            archive.extractall(path=output_dir)

        print("Extracted:", file)

Extracted: test.csv.7z
Extracted: stores.csv.7z
Extracted: items.csv.7z
Extracted: holidays_events.csv.7z
Extracted: transactions.csv.7z
Extracted: train.csv.7z
Extracted: oil.csv.7z
Extracted: sample_submission.csv.7z


## 2. Load CSV files into the Dataframe

In [3]:

train = pd.read_csv(output_dir+"/train.csv")
test = pd.read_csv(output_dir+"/test.csv")
stores = pd.read_csv(output_dir +"/stores.csv")
items = pd.read_csv(output_dir +"/items.csv")
holidays = pd.read_csv(output_dir +"/holidays_events.csv")
# basic inspection of the data
print(train.shape)
print(train.head())

train.info()
train.isnull().sum()

/tmp/ipykernel_57/2145078197.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(output_dir+"/train.csv")


(125497040, 6)
   id        date  store_nbr  item_nbr  unit_sales onpromotion
0   0  2013-01-01         25    103665         7.0         NaN
1   1  2013-01-01         25    105574         1.0         NaN
2   2  2013-01-01         25    105575         2.0         NaN
3   3  2013-01-01         25    108079         1.0         NaN
4   4  2013-01-01         25    108701         1.0         NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   item_nbr     int64  
 4   unit_sales   float64
 5   onpromotion  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 5.6+ GB


id                    0
date                  0
store_nbr             0
item_nbr              0
unit_sales            0
onpromotion    21657651
dtype: int64

## 3. Clean the Train and Test Dataset

In [4]:


# convert date columns to datetime
train['date'] = pd.to_datetime(train['date'])
# get data last 1 year
cutoff = train['date'].max() - pd.Timedelta(days=365)
train = train[train['date'] >= cutoff]
# handle negative sales 
train['unit_sales'] = train['unit_sales'].apply(lambda x: max(x, 0))
# log-transform sales to reduce skewness
train['unit_sales'] = np.log1p(train['unit_sales'])

# convert date columns to datetime
test['date'] = pd.to_datetime(test['date'])


## 4. Clean stores Dataset

In [5]:
# explore table
print(stores.info())
stores.isnull().sum()
display(stores.head())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB
None


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


## 5. Clean items Dataset



In [6]:
# explore table
print(items.info())
display(items.head())
# check for missing values in the other datasets
print(items.isnull().sum())
# convert categorical columns to category dtype for memory efficiency
items['family'] = items['family'].astype('category')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   item_nbr    4100 non-null   int64 
 1   family      4100 non-null   object
 2   class       4100 non-null   int64 
 3   perishable  4100 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 128.3+ KB
None


,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


item_nbr      0
family        0
class         0
perishable    0
dtype: int64


## 6. Clean holidays_events Dataset


In [7]:
# explore table'
print(holidays.info())
display(holidays.head())
print(holidays.isnull().sum())
print(f"Duplicated days: {holidays['date'].duplicated().sum()}")
# convert date columns to datetime
holidays['date'] = pd.to_datetime(holidays['date'])
# keep only relevant holidays
holidays = holidays[holidays['transferred'] == False]
# simplify holiday
holidays['is_holiday'] = 1
holidays = holidays[['date', 'is_holiday']]
display(holidays.head())
# drop duplicate holidays
holidays = holidays.drop_duplicates('date')



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         350 non-null    object
 1   type         350 non-null    object
 2   locale       350 non-null    object
 3   locale_name  350 non-null    object
 4   description  350 non-null    object
 5   transferred  350 non-null    bool  
dtypes: bool(1), object(5)
memory usage: 14.1+ KB
None


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64
Duplicated days: 38


,date,is_holiday
0,2012-03-02,1
1,2012-04-01,1
2,2012-04-12,1
3,2012-04-14,1
4,2012-04-21,1


## 7. Merge train and test into one dataframe and merge it with other Datasets

In [8]:
import pandas as pd




# 1. Ensure both datasets have same columns

# test usually has no unit_sales column
if 'unit_sales' not in test.columns:
    test['unit_sales'] = pd.NA


# 2. Concatenate train + test

full = pd.concat(
    [train, test],
    ignore_index=True,
    sort=False
)




# 3. Merge stores dataset


full = full.merge(
    stores,
    on='store_nbr',
    how='left'
)


# 4. Merge items dataset


full = full.merge(
    items,
    on='item_nbr',
    how='left'
)


# 5. Merge holidays dataset

full = full.merge(
    holidays,
    on='date',
    how='left'
)


# 6. Check duplicates

duplicates = full.duplicated(
    ['date', 'store_nbr', 'item_nbr']
).sum()

print("Duplicates:", duplicates)


# 7. Fill missing values


# holiday flag
full['is_holiday'] = full['is_holiday'].fillna(0)

# promotion flag
full['onpromotion'] = (
    full['onpromotion']
    .fillna("None")
)


# 10. Explore merged dataset

print(full.info())

display(full.head())

print(full.isnull().sum())

/tmp/ipykernel_57/3909097476.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full = pd.concat(


Duplicates: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40825299 entries, 0 to 40825298
Data columns (total 14 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[ns]
 2   store_nbr    int64         
 3   item_nbr     int64         
 4   unit_sales   float64       
 5   onpromotion  bool          
 6   city         object        
 7   state        object        
 8   type         object        
 9   cluster      int64         
 10  family       category      
 11  class        int64         
 12  perishable   int64         
 13  is_holiday   float64       
dtypes: bool(1), category(1), datetime64[ns](1), float64(2), int64(6), object(3)
memory usage: 3.7+ GB
None


/tmp/ipykernel_57/3909097476.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("None")


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,family,class,perishable,is_holiday
0,88042205,2016-08-15,1,103665,0.693147,False,Quito,Pichincha,D,13,BREAD/BAKERY,2712,1,1.0
1,88042206,2016-08-15,1,105574,0.693147,False,Quito,Pichincha,D,13,GROCERY I,1045,0,1.0
2,88042207,2016-08-15,1,105575,2.995732,False,Quito,Pichincha,D,13,GROCERY I,1045,0,1.0
3,88042208,2016-08-15,1,105577,0.693147,False,Quito,Pichincha,D,13,GROCERY I,1045,0,1.0
4,88042209,2016-08-15,1,105693,0.693147,False,Quito,Pichincha,D,13,GROCERY I,1034,0,1.0


id                   0
date                 0
store_nbr            0
item_nbr             0
unit_sales     3370464
onpromotion          0
city                 0
state                0
type                 0
cluster              0
family               0
class                0
perishable           0
is_holiday           0
dtype: int64


In [9]:
"""
# 7. Save final dataset
os.makedirs("/kaggle/working/cleanedCSV", exist_ok=True)
data_cleaned_path = "/kaggle/working/cleanedCSV/"
data.to_csv(data_cleaned_path +"data_cleaned.csv", index=False)
"""

'\n# 7. Save final dataset\nos.makedirs("/kaggle/working/cleanedCSV", exist_ok=True)\ndata_cleaned_path = "/kaggle/working/cleanedCSV/"\ndata.to_csv(data_cleaned_path +"data_cleaned.csv", index=False)\n'

In [10]:
"""
# Zip data to download
!zip -r hazards_results.zip /kaggle/working/cleanedCSV
from IPython.display import FileLink
FileLink(r'hazards_results.zip')
"""

"\n# Zip data to download\n!zip -r hazards_results.zip /kaggle/working/cleanedCSV\nfrom IPython.display import FileLink\nFileLink(r'hazards_results.zip')\n"

## 9. Features Engineer the full Dataset (Train+Test) Dataset


In [11]:
data = full


# SECTION 1 — DATE FEATURES

data = data.sort_values(['store_nbr', 'item_nbr', 'date'])
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['day'] = data['date'].dt.day
data['dayofweek'] = data['date'].dt.dayofweek



# SECTION 2 — CREATE LAG FEATURES


group_cols = ['store_nbr', 'item_nbr']

data['lag_7'] = (
    data.groupby(group_cols)['unit_sales']
    .shift(7)
)

data['lag_14'] = (
    data.groupby(group_cols)['unit_sales']
    .shift(14)
)


# SECTION 3 — ROLLING FEATURES 


data['rolling_mean_7'] = (
    data.groupby(group_cols)['unit_sales']
    .shift(1)
    .rolling(7)
    .mean()
)


# SECTION 4 — HIERARCHICAL MISSING VALUE STRATEGY

lag_cols = [
    'lag_7',
    'lag_14',
    'rolling_mean_7'
]


for col in lag_cols:

   
    # Layer 1: SAME STORE + SAME ITEM
   
    data[col] = data[col].fillna(
        data.groupby(['store_nbr', 'item_nbr'])[col]
        .transform('median')
    )

   
    # Layer 2: SAME ITEM
   
    data[col] = data[col].fillna(
        data.groupby('item_nbr')[col]
        .transform('median')
    )

    
    # Layer 3: SAME STORE
   
    data[col] = data[col].fillna(
        data.groupby('store_nbr')[col]
        .transform('median')
    )

   
    # Layer 4: GLOBAL MEDIAN
    
    data[col] = data[col].fillna(
        data[col].median()
    )










In [12]:
# SECTION 5 — ENCODE CATEGORICAL VARIABLES

categorical_cols = [
    'family',
    'city',
    'state',
    'type'
]

data = pd.get_dummies(
    data,
    columns=categorical_cols,
    drop_first=True
)

# SECTION 6 — SPLIT BACK TRAIN & TEST


train_fe = data[data['unit_sales'].notna()]
test_fe = data[data['unit_sales'].isna()]


# SECTION 7 — FINAL CHECKS

print("Train shape:")
print(train_fe.shape)

print("\nTest shape:")
print(test_fe.shape)

print("\nRemaining missing values in features:")
print(train_fe.isnull().sum())
print(test_fe.isnull().sum())


display(train_fe.head())
display(test_fe.head())

Train shape:
(37454835, 89)

Test shape:
(3370464, 89)

Remaining missing values in features:
id                  0
date                0
store_nbr           0
item_nbr            0
unit_sales          0
                   ..
state_Tungurahua    0
type_B              0
type_C              0
type_D              0
type_E              0
Length: 89, dtype: int64
id                        0
date                      0
store_nbr                 0
item_nbr                  0
unit_sales          3370464
                     ...   
state_Tungurahua          0
type_B                    0
type_C                    0
type_D                    0
type_E                    0
Length: 89, dtype: int64


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,cluster,class,perishable,is_holiday,...,state_Manabi,state_Pastaza,state_Pichincha,state_Santa Elena,state_Santo Domingo de los Tsachilas,state_Tungurahua,type_B,type_C,type_D,type_E
23607496,111649701,2017-04-07,1,96995,1.098612,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False
24653839,112696044,2017-04-17,1,96995,0.693147,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False
25059171,113101376,2017-04-21,1,96995,1.098612,False,13,1093,0,1.0,...,False,False,True,False,False,False,False,False,True,False
25163868,113206073,2017-04-22,1,96995,1.386294,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False
25596594,113638799,2017-04-26,1,96995,0.693147,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,cluster,class,perishable,is_holiday,...,state_Manabi,state_Pastaza,state_Pichincha,state_Santa Elena,state_Santo Domingo de los Tsachilas,state_Tungurahua,type_B,type_C,type_D,type_E
37454835,125497040,2017-08-16,1,96995,NaN,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False
37665489,125707694,2017-08-17,1,96995,NaN,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False
37876143,125918348,2017-08-18,1,96995,NaN,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False
38086797,126129002,2017-08-19,1,96995,NaN,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False
38297451,126339656,2017-08-20,1,96995,NaN,False,13,1093,0,0.0,...,False,False,True,False,False,False,False,False,True,False


# III. Train Model

## 2. Train XGboost models

In [13]:

stores = train_fe['store_nbr'].unique()

models = {}

for s in stores:

    df_s = train_fe[train_fe['store_nbr'] == s].copy()

    X = df_s.drop(columns=['unit_sales', 'date', 'id','store_nbr'])
    y = df_s['unit_sales']

    model = XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.7,
        n_jobs=2
    )

    model.fit(X, y)

    models[s] = model

## 3. Export Dataset for Power BI


### 3.1 Sales by store number

In [14]:
results_full = []

for s in test_fe['store_nbr'].unique():
    df_s = test_fe[test_fe['store_nbr'] == s].copy()
    X_test = df_s.drop(columns=['unit_sales','date','id','store_nbr'])
   

    model = models.get(s)
    if model is None:
        continue

    # convert to original sales
    df_s['predicted_sales'] = np.clip(
    np.expm1(model.predict(X_test)), 0, None)

    results_full.append(df_s)

final_df = pd.concat(results_full)
agg_df = final_df.groupby(['date','store_nbr']).agg({
    'unit_sales': 'sum',
    'predicted_sales': 'sum'
}).reset_index()


os.makedirs("/kaggle/working/PowerBICSV", exist_ok=True)
data_cleaned_path = "/kaggle/working/PowerBICSV/"
agg_df.to_csv(data_cleaned_path + "powerbi_data.csv", index=False)

### 3.2 Sales by family

In [15]:
results_full = []

for s in test_fe['store_nbr'].unique():
    df_s = test_fe[test_fe['store_nbr'] == s].copy()
    X_test = df_s.drop(columns=['unit_sales','date','id','store_nbr'])
    

    model = models.get(s)
    if model is None:
        continue

    # convert to original sales
    df_s['predicted_sales'] = np.clip(
    np.expm1(model.predict(X_test)), 0, None)

    results_full.append(df_s)

final_df = pd.concat(results_full)
family_cols = [c for c in final_df.columns if c.startswith('family_')]
final_df['family'] = final_df[family_cols].idxmax(axis=1).str.replace('family_', '')
agg_df = final_df.groupby(['date', 'store_nbr', 'family']).agg({
    'unit_sales': 'sum',
    'predicted_sales': 'sum'
}).reset_index()



data_cleaned_path = "/kaggle/working/PowerBICSV/"
agg_df.to_csv(data_cleaned_path + "powerbi_data1.csv", index=False)

In [16]:

# Zip data to download
!zip -r powerBI.zip /kaggle/working/PowerBICSV
from IPython.display import FileLink
FileLink(r'powerBI.zip')


  adding: kaggle/working/PowerBICSV/ (stored 0%)
  adding: kaggle/working/PowerBICSV/powerbi_data.csv (deflated 72%)
  adding: kaggle/working/PowerBICSV/powerbi_data1.csv (deflated 80%)


/kaggle/working/powerBI.zip

## 5. Power BI Visualization for predicted sales and actual sales by store and family product 